# eTOF time-of-flight → kinetic-energy calibration (config 1)

**Post-analysis task 1** (see `analysis/post/TASKS.md`).

## Objective
Map electron **time-of-flight (TOF, ns) → electron kinetic energy (KE, eV)**
for the config-1 electron TOF spectrometer. This calibration underpins
every downstream electron analysis (peak assignment, residual-gas
background, covariance).

## Data
| Role | Run | Notes |
|------|-----|-------|
| Argon calibration | **58826** | known Ar photolines + LMM Auger lines |
| Residual-gas background | **58825** | subtracted to isolate the Ar signal |

Both are config-1 combined H5 files (electron + ion TOF), built by
`write_h5.py` and resolved through `config.COMBINED_DIR`.

## Method
1. Load both runs; restrict to a clean signal-bunch range.
2. Build per-shot-normalised **average eTOF spectra** for Ar and for the
   residual gas (memory-efficient single-histogram over non-zero hits).
3. Subtract the residual-gas spectrum from the Ar spectrum.
4. Identify known Ar lines (photolines: KE = hν − BE; LMM Auger: fixed KE).
5. Fit the TOF→KE model  `KE(t) = (b / (t − t0))²`  to the (TOF peak, KE)
   pairs.
6. Save the calibration parameters and validate.

> This notebook is a **scaffold**: set the photon energy and the
> peak↔line assignments after inspecting the spectrum, then run top to
> bottom. It expects the combined H5 files for runs 58825 / 58826 to exist.

In [ ]:
import sys
import json
from pathlib import Path

# analysis/post -> repo root is two levels up (same layout as analysis/notebooks).
_REPO_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(_REPO_ROOT / "analysis" / "scripts"))

import numpy as np
import matplotlib.pyplot as plt

import config
from data_loading import load_data

try:
    from scipy.signal import find_peaks
    from scipy.optimize import curve_fit
    _HAVE_SCIPY = True
except Exception:
    _HAVE_SCIPY = False
    print("scipy unavailable - peak-finding / nonlinear-fit cells will be skipped.")

%matplotlib inline

## Parameters

Set the run numbers, the **photon energy of the Argon run** (needed for
photoline KE = hν − BE), the clean signal-bunch range (from the
diagnostics notebook for these runs), and the TOF histogram binning.

In [ ]:
# --- Runs --------------------------------------------------------------
RUN_AR = 58826   # Argon calibration
RUN_BG = 58825   # residual-gas background

# Combined-H5 paths. Default to COMBINED_DIR/run<N>.h5; override if needed.
AR_H5 = config.COMBINED_DIR / f"run{RUN_AR}.h5"
BG_H5 = config.COMBINED_DIR / f"run{RUN_BG}.h5"

# --- Photon energy of the Argon run (eV) -------------------------------
# REQUIRED for photoline KE = hv - BE. Read from the logbook / mpe.
PHOTON_ENERGY_EV = None   # e.g. 285.0

# --- Signal bunch range (half-open) ------------------------------------
# Determine from the diagnostics notebook for these runs.
SIGNAL_BUNCH_RANGE = (10, 40)

# --- TOF histogram binning (ns) ----------------------------------------
# Span the eTOF window; refine after a first look at the spectrum.
TOF_MIN, TOF_MAX, TOF_BIN = 0.0, 2000.0, 1.0
tof_edges = np.arange(TOF_MIN, TOF_MAX + TOF_BIN, TOF_BIN)
tof_cent  = 0.5 * (tof_edges[:-1] + tof_edges[1:])

# --- Normalisation -----------------------------------------------------
# "per_shot": counts per (train, bunch) shot.
# "per_uJ"  : counts per shot per uJ of GMD (fluence-normalised).
NORMALISE = "per_shot"

# --- Background scaling -------------------------------------------------
# Residual-gas pressure may differ between runs; tune if the subtraction
# over/under-shoots in a known-empty TOF region.
BG_SCALE = 1.0

# --- Resolve / check ---------------------------------------------------
for label, p in [("Ar", AR_H5), ("background", BG_H5)]:
    if not Path(p).exists():
        raise FileNotFoundError(
            f"{label} combined H5 not found: {p}\n"
            f"Build it with write_h5.py (config 1) or set the path above."
        )
print(f"Ar run {RUN_AR}: {AR_H5}")
print(f"BG run {RUN_BG}: {BG_H5}")
print(f"signal bunches [{SIGNAL_BUNCH_RANGE[0]}, {SIGNAL_BUNCH_RANGE[1]}), "
      f"TOF [{TOF_MIN}, {TOF_MAX}] ns / {TOF_BIN} ns bins, normalise={NORMALISE}")

## 1. Load both runs

`load_data(..., config=1)` removes bad trains and trims edges. We only
need `tofs_e` and `gmd` here.

In [ ]:
b0, b1 = SIGNAL_BUNCH_RANGE

ar = load_data(str(AR_H5), config=1)
bg = load_data(str(BG_H5), config=1)

for name, d in [("Ar", ar), ("background", bg)]:
    if d.tofs_e is None:
        raise RuntimeError(f"{name} run has no tofs_e - is it a config-1 file?")
    print(f"{name:>10}: n_trains={d.n_trains}, m={d.n_bunches}, "
          f"max_ecounts={d.tofs_e.shape[-1]}")
    if not (0 <= b0 < b1 <= d.n_bunches):
        raise ValueError(
            f"{name}: SIGNAL_BUNCH_RANGE {SIGNAL_BUNCH_RANGE} outside [0, {d.n_bunches}]"
        )

## 2. Average eTOF spectra (memory-efficient)

eTOF arrays are zero-padded: real hits are the non-zero values. We slice
the signal bunches, drop the padding, and histogram **all** hits in one
pass (no per-shot loop). Per-shot (or per-µJ) normalisation puts the Ar
and residual-gas spectra on the same footing so they subtract directly.

In [ ]:
def average_etof(data, bunch_range, edges, normalise="per_shot"):
    """Average eTOF spectrum over a signal bunch range.

    Excludes zero-padding, histograms all hits in a single pass, and
    normalises so spectra from different runs are comparable.
    Returns (spectrum, n_shots, total_gmd).
    """
    lo, hi = bunch_range
    hits = data.tofs_e[:, lo:hi, :]            # (n_trains, n_sig, max_hits)
    n_shots = hits.shape[0] * hits.shape[1]    # number of (train, bunch) shots
    valid = hits[hits > 0]                     # drop zero-padding
    counts, _ = np.histogram(valid, bins=edges)
    counts = counts.astype(np.float64)

    gmd_sig = np.asarray(data.gmd[:, lo:hi], dtype=np.float64)
    total_gmd = np.nansum(gmd_sig)

    if normalise == "per_shot":
        spectrum = counts / max(n_shots, 1)
    elif normalise == "per_uJ":
        spectrum = counts / total_gmd if total_gmd > 0 else counts
    else:
        raise ValueError(f"unknown NORMALISE={normalise!r}")
    return spectrum, n_shots, total_gmd

ar_spec, ar_n, ar_g = average_etof(ar, SIGNAL_BUNCH_RANGE, tof_edges, NORMALISE)
bg_spec, bg_n, bg_g = average_etof(bg, SIGNAL_BUNCH_RANGE, tof_edges, NORMALISE)

print(f"Ar: {ar_n} shots, total GMD {ar_g:.3g} uJ")
print(f"BG: {bg_n} shots, total GMD {bg_g:.3g} uJ")

## 3. Background subtraction

Both spectra are per-shot (or per-µJ) normalised, so they subtract
directly. `BG_SCALE` corrects for a residual-gas pressure difference
between the two runs — tune it so the subtraction is flat in a TOF
region where Ar has no signal.

In [ ]:
ar_sub = ar_spec - BG_SCALE * bg_spec

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                               constrained_layout=True)
ax1.plot(tof_cent, ar_spec, lw=0.8, label=f"Ar (run {RUN_AR})")
ax1.plot(tof_cent, BG_SCALE * bg_spec, lw=0.8, alpha=0.8,
         label=f"residual gas (run {RUN_BG}) x{BG_SCALE:g}")
ax1.set_ylabel(f"counts ({NORMALISE})")
ax1.legend(); ax1.set_title("Raw Ar vs residual-gas eTOF spectra")

ax2.plot(tof_cent, ar_sub, lw=0.8, color="k", label="Ar - background")
ax2.axhline(0, color="r", lw=0.5)
ax2.set_xlabel("eTOF (ns)"); ax2.set_ylabel(f"counts ({NORMALISE})")
ax2.legend(); ax2.set_title("Background-subtracted Ar eTOF spectrum")
plt.show()

## 4. Identify known Argon lines

Inspect the subtracted spectrum above and assign TOF peaks to known Ar
lines. Two useful families:

- **Photolines** — KE depends on photon energy: `KE = hν − BE`.

  | line | BE (eV) |
  |------|---------|
  | Ar 2p₃/₂ | 248.4 |
  | Ar 2p₁/₂ | 250.6 |
  | Ar 3s    | 29.24 |
  | Ar 3p    | 15.76 |

- **L₂,₃M₂,₃M₂,₃ (LMM) Auger** — *fixed* KE, independent of hν; the
  strongest group sits near **KE ≈ 200–206 eV** (e.g. the ¹D₂ line at
  ~203 eV). An excellent calibrant precisely because it does not move
  with hν.

The cell below runs a peak finder to suggest candidate TOF peaks; then
fill in `ASSIGNMENTS` by hand as `(tof_peak_ns, KE_eV, label)`.

In [ ]:
# Suggest candidate peaks (tune prominence to the data).
if _HAVE_SCIPY:
    pk, props = find_peaks(ar_sub, prominence=np.nanmax(ar_sub) * 0.02)
    order = np.argsort(props["prominences"])[::-1]
    print("candidate peaks (most prominent first):")
    print(f"{'TOF (ns)':>10} {'counts':>12} {'prominence':>12}")
    for i in order[:25]:
        print(f"{tof_cent[pk[i]]:>10.2f} {ar_sub[pk[i]]:>12.4g} "
              f"{props['prominences'][i]:>12.4g}")
else:
    print("scipy unavailable - read peak TOFs off the plot manually.")

def photoline_ke(be_eV):
    """KE of an Ar photoline of binding energy be_eV at PHOTON_ENERGY_EV."""
    if PHOTON_ENERGY_EV is None:
        raise ValueError("set PHOTON_ENERGY_EV to use photoline_ke()")
    return PHOTON_ENERGY_EV - be_eV

# --- FILL IN after inspecting the spectrum ----------------------------
# Each entry: (tof_peak_ns, known_KE_eV, label). Example (edit & uncomment):
# ASSIGNMENTS = [
#     (XXX.X, photoline_ke(248.4), "Ar 2p3/2"),
#     (XXX.X, photoline_ke(250.6), "Ar 2p1/2"),
#     (XXX.X, 203.0,               "Ar LMM (1D2)"),
# ]
ASSIGNMENTS = []
assert len(ASSIGNMENTS) >= 2, (
    "need at least 2 (TOF, KE) pairs to fit; 3+ recommended for a residual check"
)

## 5. Fit the TOF → KE calibration

Single-field TOF spectrometer relation:

$$ t = t_0 + \frac{b}{\sqrt{KE}} \quad\Longleftrightarrow\quad KE(t) = \left(\frac{b}{t - t_0}\right)^2 $$

We linearise in $KE^{-1/2}$ for a robust first estimate, then refine the
forward model with a nonlinear fit.

In [ ]:
# Pull the assigned (TOF, KE) pairs.
asg = np.array([(t, ke) for t, ke, _ in ASSIGNMENTS], dtype=np.float64)
t_pk, ke_known = asg[:, 0], asg[:, 1]

# Linear estimate: t = t0 + b * KE^{-1/2}  (slope = b, intercept = t0).
x = 1.0 / np.sqrt(ke_known)
b_lin, t0_lin = np.polyfit(x, t_pk, 1)
print(f"linear estimate: t0 = {t0_lin:.3f} ns, b = {b_lin:.3f}")

def t_of_ke(ke, t0, b):
    """Forward model: TOF (ns) for a kinetic energy KE (eV)."""
    return t0 + b / np.sqrt(ke)

# Nonlinear refine.
if _HAVE_SCIPY:
    (t0, b), _ = curve_fit(t_of_ke, ke_known, t_pk, p0=(t0_lin, b_lin))
else:
    t0, b = t0_lin, b_lin
print(f"fitted: t0 = {t0:.4f} ns, b = {b:.4f}  ->  KE(t) = (b/(t-t0))^2")

# Residuals at the calibration points.
t_pred = t_of_ke(ke_known, t0, b)
resid = t_pk - t_pred
for (t, ke, lbl), tp, r in zip(ASSIGNMENTS, t_pred, resid):
    print(f"  {lbl:>16}: KE={ke:8.2f} eV  TOF={t:8.2f}  pred={tp:8.2f}  d={r:+.3f} ns")
rms = float(np.sqrt(np.mean(resid ** 2)))
print(f"RMS TOF residual: {rms:.3f} ns")

# Calibration curve.
fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
ke_grid = np.linspace(ke_known.min() * 0.6, ke_known.max() * 1.2, 400)
ax.plot(t_of_ke(ke_grid, t0, b), ke_grid, "-", label="fit  KE=(b/(t-t0))^2")
ax.plot(t_pk, ke_known, "o", label="assigned lines")
for t, ke, lbl in ASSIGNMENTS:
    ax.annotate(lbl, (t, ke), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("eTOF (ns)"); ax.set_ylabel("KE (eV)")
ax.legend(); ax.set_title("TOF -> KE calibration")
plt.show()

## 6. Apply the calibration and save

Plot the Ar spectrum on a KE axis. Because the axis is nonlinear, divide
the per-TOF-bin counts by `|dKE/dt|` to get a per-eV density (Jacobian
correction). Then persist the calibration parameters so every other post
notebook loads the same one.

In [ ]:
def tof_to_ke(t, t0=t0, b=b):
    """Calibrated electron kinetic energy (eV) for a TOF (ns).

    Valid for t > t0; NaN otherwise. dKE/dt is large at small KE, so KE
    resolution degrades for slow (long-TOF) electrons.
    """
    t = np.asarray(t, dtype=np.float64)
    return np.where(t > t0, (b / (t - t0)) ** 2, np.nan)

# Convert the subtracted Ar spectrum onto a KE axis (Jacobian-corrected).
ke_axis = tof_to_ke(tof_cent)
dke_dt = np.gradient(ke_axis, tof_cent)
with np.errstate(invalid="ignore", divide="ignore"):
    spec_ke = ar_sub / np.abs(dke_dt)   # counts per eV

good = np.isfinite(ke_axis)
fig, ax = plt.subplots(figsize=(11, 4), constrained_layout=True)
ax.plot(ke_axis[good], spec_ke[good], lw=0.8, color="k")
for t, ke, lbl in ASSIGNMENTS:
    ax.axvline(ke, color="r", lw=0.5, ls="--")
    ax.annotate(lbl, (ke, ax.get_ylim()[1]), fontsize=8,
                rotation=90, va="top", ha="right")
ax.set_xlabel("electron KE (eV)"); ax.set_ylabel("counts / eV")
ax.set_title(f"Ar eTOF on KE axis (run {RUN_AR})")
plt.show()

# Persist the calibration.
calib_dir = Path(_REPO_ROOT) / "analysis" / "post" / "calibration"
calib_dir.mkdir(parents=True, exist_ok=True)
calib = {
    "model": "KE = (b / (t - t0))**2",
    "t0_ns": float(t0),
    "b": float(b),
    "run_ar": int(RUN_AR),
    "run_bg": int(RUN_BG),
    "photon_energy_eV": PHOTON_ENERGY_EV,
    "signal_bunch_range": list(SIGNAL_BUNCH_RANGE),
    "assignments": [[float(t), float(ke), lbl] for t, ke, lbl in ASSIGNMENTS],
    "rms_tof_residual_ns": rms,
}
calib_path = calib_dir / f"etof_tof_to_ke_run{RUN_AR}.json"
with open(calib_path, "w") as fh:
    json.dump(calib, fh, indent=2)
print(f"saved calibration -> {calib_path}")

## 7. Validation

- Check the **RMS TOF residual** is small versus your bin width and that
  no single line dominates it.
- Confirm the **fixed-KE LMM Auger** line lands at the same KE if you
  refit at a different photon energy (it must not move with hν).
- Use the leave-one-out cell below: drop each line, refit, predict it.
- Sanity-check the KE-axis spectrum — photolines should sit at `hν − BE`.

To reuse the calibration in another post notebook:

```python
import json, numpy as np
c = json.load(open("analysis/post/calibration/etof_tof_to_ke_run58826.json"))
tof_to_ke = lambda t: (c["b"] / (np.asarray(t) - c["t0_ns"])) ** 2
```

> **TODO (TASKS.md backlog):** decide whether `tof_to_ke` should be
> promoted into `analysis/scripts/processing.py` (loading params from this
> JSON) so all post notebooks share one implementation.

In [ ]:
# Leave-one-out: refit without each line and predict its TOF.
if _HAVE_SCIPY and len(ASSIGNMENTS) >= 3:
    print("leave-one-out TOF prediction:")
    for k in range(len(ASSIGNMENTS)):
        keep = [j for j in range(len(ASSIGNMENTS)) if j != k]
        (t0_k, b_k), _ = curve_fit(
            t_of_ke, ke_known[keep], t_pk[keep], p0=(t0_lin, b_lin)
        )
        pred = t_of_ke(ke_known[k], t0_k, b_k)
        lbl = ASSIGNMENTS[k][2]
        print(f"  {lbl:>16}: actual {t_pk[k]:8.2f}  predicted {pred:8.2f}  "
              f"d={t_pk[k] - pred:+.3f} ns")
else:
    print("need scipy and >=3 assignments for leave-one-out validation.")